# Reasoning Shortcuts in Neurosymbolic Learning

Neurosymbolic (NeSy) models split a task into two halves:

- a **neural** half that extracts *concepts* from raw input (e.g. "this image is the digit 3", "a pedestrian is present"),
- a **symbolic** half that combines those concepts with a fixed rule to produce the final label (e.g. "3 + 4 = 7", "pedestrian present -> stop").

Only the final label is ever supervised during training. The concepts in between are **latent** -- nobody ever tells the model "this is a 3." This notebook is about what goes wrong when that's the only signal the model gets.

**Central takeaway:** *a model can learn the right rule over concepts with unintended semantics.*

## A precise definition

Following Marconato, Teso, Vergari, and Passerini (2023, NeurIPS -- *"Not All Neuro-Symbolic Concepts Are Created Equal,"* [arXiv:2305.19951](https://arxiv.org/abs/2305.19951)), a **reasoning shortcut (RS)** is a learned concept distribution that:

1. achieves **maximal (or near-maximal) likelihood on the training objective** -- it is not an undertrained or badly optimized model, it is one of the *best* solutions available, and
2. **does not match the ground-truth concept distribution** -- its semantics are misaligned with what we intended, even though its behavior is optimal.

Both conditions matter, and this notebook checks both, every time. A model with low concept accuracy that *hasn't* reached its best achievable task performance is not evidence of a reasoning shortcut -- it's evidence of an optimization problem (undertraining, a bad learning rate, a badly conditioned loss). We'll show a couple of examples where an experiment can look like it's demonstrating a shortcut, but is really just demonstrating that a network hasn't converged yet. Distinguishing the two is most of the skill this notebook is trying to build.

## Four causes, four demonstrations

The paper traces reasoning shortcuts to four sources, and this notebook gives the first three a concrete, verified demonstration, then applies the idea to a more realistic setting:

1. **Prior knowledge** -- the symbolic rule itself may permit several different concept assignments to produce the same answer. (Section 2: exhaustive XOR.)
2. **Architecture** -- how the neural half is wired can independently create or foreclose ambiguity, even on identical data. (Section 3: the same MNIST-Addition task, with two different encoder architectures.)
3. **Data structure / selection bias** -- if some combinations of concepts are never observed together, the model has no way to tell which one is doing the work. (Section 4, briefly, via the paper's own Add+Multiply result.)
4. **Learning objective** -- if the loss only rewards correct final labels, the model has no incentive to prefer the *true* concepts over any other concept assignment that also works. (True of every "no concept supervision" run below -- and the thing every mitigation in this notebook pushes back against.)

Section 5 then takes the idea somewhere more realistic: a driving scenario with confounded sensor data. That section is best read as an **application and transfer exercise**, not a fourth controlled demonstration on the same footing as Sections 2-4 -- we'll be explicit there about exactly what it does and doesn't establish, since it sits closer to the boundary with ordinary spurious-correlation shortcut learning than the earlier sections do.

One terminology note we'll keep coming back to: in the architectures below, the symbolic rule is applied directly to the network's own concept beliefs, so **"constraint satisfaction" -- does the output obey the rule? -- is guaranteed by construction, not evidence of anything.** It tells you the *plumbing* works, never whether the concepts flowing through it mean what you think they mean.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from sklearn.metrics import f1_score, confusion_matrix
from torchvision import datasets, transforms

# Fixing the random seeds makes every run of this notebook reproducible.
torch.manual_seed(0)
np.random.seed(0)


## 1. Prior knowledge: exhaustive 3-way XOR

Before touching any images or neural networks, we can prove the underdetermination problem outright, with eight rows of a table.

This follows the paper's own XOR construction: three binary concepts `G1, G2, G3`, and a label `Y = G1 XOR G2 XOR G3`. There are only eight possible inputs, and we're about to show *all eight* -- there is no missing data, no held-out combination, nothing more the model could ever be shown. The paper states this outcome directly: *"it is impossible to pin down the ground-truth distribution even if all possible combinations of inputs are observed."*

Why three concepts rather than two? With two concepts, flipping *both* of them preserves 2-way XOR (`a XOR b == (NOT a) XOR (NOT b)`), which is already a clean illustration -- but with three, there turn out to be **three different ways** to flip a pair of concepts and still preserve the label, giving a richer picture of just how many equally-valid wrong answers coexist with the one right one.


In [ ]:
# The exhaustive, ground-truth table: every one of the 8 possible inputs, correctly labeled.
xor_table = pd.DataFrame(
    [(g1, g2, g3) for g1 in (0, 1) for g2 in (0, 1) for g3 in (0, 1)],
    columns=["g1", "g2", "g3"],
)
xor_table["label"] = xor_table.g1 ^ xor_table.g2 ^ xor_table.g3
xor_table


In [ ]:
# Flipping any TWO of the three concepts preserves the label on every row.
# Flipping all THREE does not. Both are checked below, over the full,
# exhaustive 8-row table -- there is no ninth row that could break the tie.
import itertools

for concepts_to_flip in itertools.combinations(["g1", "g2", "g3"], 2):
    flipped = xor_table.copy()
    for concept in concepts_to_flip:
        flipped[concept] = 1 - flipped[concept]
    flipped_label = flipped.g1 ^ flipped.g2 ^ flipped.g3
    label_preserved = (flipped_label == xor_table.label).all()
    both_concepts_wrong = all((flipped[c] != xor_table[c]).all() for c in concepts_to_flip)
    print(f"flip {concepts_to_flip}: label preserved on all 8 rows = {label_preserved}, "
          f"both flipped concepts wrong on every row = {both_concepts_wrong}")

flipped_all = xor_table.copy()
for concept in ["g1", "g2", "g3"]:
    flipped_all[concept] = 1 - flipped_all[concept]
flipped_all_label = flipped_all.g1 ^ flipped_all.g2 ^ flipped_all.g3
print(f"\nflip all three: label preserved on all 8 rows = {(flipped_all_label == xor_table.label).all()}")


Three distinct pairwise flips, each one wrong about **both** of its flipped concepts on **every single row**, and each one reproducing the correct label on **every single row** -- with full, exhaustive coverage of the input space. That's four equally label-optimal solutions in total (the correct one, plus these three), not one hidden edge case.

This is the cleanest possible illustration of cause 1 (prior knowledge): the rule `G1 XOR G2 XOR G3` simply does not contain enough information to distinguish these four concept assignments from each other, no matter how much data you show it. Keep this "flip a symmetry, keep the label" trick in mind -- Section 3 looks for (and fails to find, in one specific architecture) an analogous trick for real images.


## 2. Architecture: MNIST-Addition, entangled vs. disentangled

The classic empirical setting: two MNIST digit images go in, a neural network guesses each digit (the *concept*), and a fixed symbolic rule adds them together to get a sum (the *label*). Only the sum is ever supervised.

Here's the question this section actually asks, and it's about **architecture**, not data: does it matter whether the network looks at the two images *independently* or *jointly*?

- A **disentangled** encoder applies the exact same small classifier to each image, one at a time, with no way to see the other image. Whatever it decides about an image can't depend on what it happened to be paired with.
- An **entangled** encoder feeds both images into one shared trunk at once, with two output heads. It's free to let its answer for image A depend on whatever image B happened to be.

The paper reports that disentanglement, on its own, drives reasoning-shortcut frequency to roughly zero under full data coverage. We'll build both architectures, train both on the *same* exhaustive dataset (every one of the 100 possible digit-pair combinations, covered many times over), and check both halves of the definition from Section 0: does each one reach near-ceiling task accuracy, and if so, does its concept accuracy match?


In [ ]:
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())


In [ ]:
def build_exhaustive_pairs(mnist_dataset, replicas_per_combo=20, seed=0):
    """Builds a training set that covers EVERY one of the 100 possible
    (digit1, digit2) combinations, each realized by several different
    actual images -- true exhaustive coverage, not just a large random
    sample that happens to miss a few combinations.
    """
    rng = np.random.default_rng(seed)
    labels = mnist_dataset.targets.numpy()
    images_by_digit = {digit: np.where(labels == digit)[0] for digit in range(10)}
    pairs = []
    for digit1 in range(10):
        for digit2 in range(10):
            for _ in range(replicas_per_combo):
                pairs.append((rng.choice(images_by_digit[digit1]), rng.choice(images_by_digit[digit2])))
    return np.array(pairs)


class DigitPairDataset(torch.utils.data.Dataset):
    """Wraps a fixed list of (index1, index2) image pairs, labeling
    each with the true sum. The individual digit labels are kept only
    so *we* can check the model's hidden concepts -- the model never
    sees them.
    """

    def __init__(self, mnist_dataset, pairs):
        self.mnist = mnist_dataset
        self.indices = pairs

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i, j = self.indices[idx]
        image1, digit1 = self.mnist[i]
        image2, digit2 = self.mnist[j]
        return image1, image2, digit1, digit2, digit1 + digit2


exhaustive_pairs = build_exhaustive_pairs(mnist_train, replicas_per_combo=20)
exhaustive_dataset = DigitPairDataset(mnist_train, exhaustive_pairs)
print(f"{len(exhaustive_pairs)} training pairs, covering all 100 (digit1, digit2) combinations")


In [ ]:
class DisentangledEncoder(nn.Module):
    """Applies the SAME small CNN independently to each image. Whatever
    this network decides about an image cannot depend on what it was
    paired with -- there is no path for that information to reach it.
    """

    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(32 * 7 * 7, 10)

    def forward(self, image1, image2):
        digit1_probs = F.softmax(self.classifier(self.conv(image1).flatten(1)), dim=-1)
        digit2_probs = F.softmax(self.classifier(self.conv(image2).flatten(1)), dim=-1)
        return digit1_probs, digit2_probs


class EntangledEncoder(nn.Module):
    """Stacks both images into one 2-channel input and processes them
    with a SINGLE shared trunk before splitting into two output heads.
    This network CAN let its belief about image1 depend on image2 --
    there's a direct path for that information to flow through.
    """

    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head1 = nn.Linear(32 * 7 * 7, 10)
        self.head2 = nn.Linear(32 * 7 * 7, 10)

    def forward(self, image1, image2):
        joint_features = self.conv(torch.cat([image1, image2], dim=1)).flatten(1)
        return F.softmax(self.head1(joint_features), dim=-1), F.softmax(self.head2(joint_features), dim=-1)


def symbolic_sum_distribution(digit1_probs, digit2_probs):
    """The symbolic half, identical for both architectures above:
    given a belief over digit1 and a belief over digit2, computes a
    belief over their sum (0-18) via the fixed addition rule. No
    learned parameters -- and note that whatever this function
    outputs, it is BY CONSTRUCTION a valid distribution over sums
    consistent with the rule. That's the "constraint satisfaction is
    architectural" point from the introduction.
    """
    batch_size = digit1_probs.shape[0]
    joint_probs = digit1_probs.unsqueeze(2) * digit2_probs.unsqueeze(1)
    sum_probs = torch.zeros(batch_size, 19, device=digit1_probs.device)
    for digit1 in range(10):
        for digit2 in range(10):
            sum_probs[:, digit1 + digit2] += joint_probs[:, digit1, digit2]
    return sum_probs


In [ ]:
def train_addition_model(encoder_cls, dataset, epochs, lr=1e-3, batch_size=64):
    """Trains either encoder architecture using only the sum label."""
    encoder = encoder_cls()
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        for image1, image2, digit1, digit2, digit_sum in loader:
            digit1_probs, digit2_probs = encoder(image1, image2)
            sum_probs = symbolic_sum_distribution(digit1_probs, digit2_probs)
            loss = F.nll_loss(torch.log(sum_probs + 1e-8), digit_sum)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return encoder


def evaluate_addition_model(encoder, dataset):
    """Checks BOTH halves of the reasoning-shortcut definition: is task
    accuracy near its ceiling, and does concept accuracy match it?
    """
    loader = torch.utils.data.DataLoader(dataset, batch_size=256)
    true_digits, pred_digits, true_sums, pred_sums = [], [], [], []

    encoder.eval()
    with torch.no_grad():
        for image1, image2, digit1, digit2, digit_sum in loader:
            digit1_probs, digit2_probs = encoder(image1, image2)
            pred_sum = symbolic_sum_distribution(digit1_probs, digit2_probs).argmax(dim=1)
            true_digits.extend(digit1.tolist() + digit2.tolist())
            pred_digits.extend(digit1_probs.argmax(dim=1).tolist() + digit2_probs.argmax(dim=1).tolist())
            true_sums.extend(digit_sum.tolist())
            pred_sums.extend(pred_sum.tolist())
    encoder.train()

    return {
        "task_accuracy": float(np.mean(np.array(true_sums) == np.array(pred_sums))),
        "concept_f1": float(f1_score(true_digits, pred_digits, average="macro")),
        "concept_confusion": confusion_matrix(true_digits, pred_digits, labels=list(range(10))),
    }


In [ ]:
# Three random seeds per architecture. Reasoning shortcuts are ALTERNATIVE
# optima -- different initializations can land in different solutions --
# so a single run risks looking like a fluke (or like it "broke") to a
# learner who reruns the notebook and sees different numbers.
architecture_results = []
for architecture_name, encoder_cls in [("disentangled", DisentangledEncoder), ("entangled", EntangledEncoder)]:
    for seed in [0, 1, 2]:
        torch.manual_seed(seed)
        encoder = train_addition_model(encoder_cls, exhaustive_dataset, epochs=30)
        metrics = evaluate_addition_model(encoder, exhaustive_dataset)
        architecture_results.append({
            "architecture": architecture_name, "seed": seed,
            "task_accuracy": metrics["task_accuracy"], "concept_f1": metrics["concept_f1"],
        })
        if architecture_name == "entangled" and seed == 0:
            entangled_example_metrics = metrics  # keep one example around for the confusion matrix below
        if architecture_name == "disentangled" and seed == 0:
            disentangled_example_metrics = metrics

pd.DataFrame(architecture_results)


Look at the `disentangled` rows first: task accuracy and concept F1 should sit close together, both high, across all three seeds. Nothing here should suggest a shortcut, and that's the point -- per the paper's finding, disentanglement alone is doing real work.

Now the `entangled` rows: task accuracy should reach (or nearly reach) **1.000** -- literally the maximum possible, not just "high" -- while concept F1 sits far lower, and consistently so across seeds. That combination is exactly the two-part definition from the introduction: maximal task performance, wrong concept semantics. Nothing about this network's training signal ever required it to recover the real digit identities, because it had another, easier-to-reach route to a perfect score: let head1's and head2's answers about the SAME image differ depending on which pairing that image showed up in, using the joint trunk's access to both images to find a self-consistent (but semantically arbitrary) sum-preserving encoding instead.


In [ ]:
confusion = entangled_example_metrics["concept_confusion"]
# Row-normalize: raw counts make the diagonal's large values dominate the
# color scale, which can make a genuinely ~50% error rate LOOK clean just
# because the errors are individually small compared to the diagonal. Row
# proportions show each true digit's error rate honestly.
row_normalized_confusion = confusion / confusion.sum(axis=1, keepdims=True)

figure, axis = plt.subplots(figsize=(6.5, 6))
image = axis.imshow(row_normalized_confusion, cmap="Blues", vmin=0, vmax=1)
for true_digit in range(10):
    for predicted_digit in range(10):
        value = row_normalized_confusion[true_digit, predicted_digit]
        axis.text(predicted_digit, true_digit, f"{value:.2f}", ha="center", va="center",
                  fontsize=8, color="white" if value > 0.5 else "#333")
axis.set_xlabel("Predicted digit")
axis.set_ylabel("True digit")
axis.set_title("Entangled encoder: digit concept confusion\n(fraction of each true digit's images, by predicted digit)")
axis.set_xticks(range(10))
axis.set_yticks(range(10))
figure.colorbar(image, ax=axis, label="fraction of row")
plt.show()


Look at the off-diagonal cells: each true digit's errors should spread out almost *evenly* across the other nine predicted digits, roughly 0.05 apiece, rather than concentrating on one or two commonly-confused digits. That's a meaningfully different (and more precise) story than "the model learned a consistent alternate labeling for digits": there is no stable alternate mapping to point to. The entangled network's two output heads only ever had to agree with each other on the *specific pair* of images they were jointly shown -- nothing in the training signal ever asked "would you still say this about this image if it had been paired with a different one?" So an individual image's predicted digit isn't a fixed property of that image at all; it depends on whichever image it happened to be paired with that time, which is exactly why, averaged across many different pairings, it looks close to uniformly spread over the wrong classes.

We'd call this a genuine reasoning shortcut, not an optimization failure, specifically *because* its task accuracy already sat at the ceiling in the table above -- more training time would not have changed this outcome. Compare that to Section 4 below, where a demo can look similar at first glance but turns out to be undertrained instead.


## 3. A caution: don't mistake undertraining for a reasoning shortcut

It's worth dwelling on this failure mode because it's an easy one to fall into -- including, during earlier drafts of this notebook, for us. Low concept accuracy alongside so-so task accuracy is ambiguous: it's consistent with a genuine reasoning shortcut, but it's equally consistent with a network that simply hasn't converged.

The fix is always the same: check whether task accuracy is at (or extremely close to) its ceiling first. If it isn't, any conclusion about concept quality is premature -- you're likely looking at an optimization problem, and the right next step is more training or a better learning rate, not a rewrite of the experiment. Only once task accuracy has plateaued at its best achievable value does a persistently low concept score become informative.

### Multi-task learning as a mitigation: addition + multiplication

The paper's own mitigation experiment for a data-support-driven reasoning shortcut is worth citing directly rather than re-deriving from scratch here, because it isolates the mechanism so cleanly. Under a restricted MNIST-AddMul setup (the model only ever observes a handful of specific digit-pair combinations, not the full 100), they report:

| Setup | Concept F1 |
|---|---|
| Addition supervision alone | ~0% |
| Addition **and** multiplication supervision, jointly | ~99.8% |

Why would adding a second symbolic task fix a first one? Because of a simple, checkable fact: knowing only the **sum** of two unknown digits leaves many candidate pairs consistent with it (`3+6`, `4+5`, `2+7`, ... all sum to 9) -- but knowing **both the sum and the product** pins the pair down to essentially one answer, because `d1` and `d2` become the two roots of `x^2 - (sum)x + (product) = 0`. Multiplying supervision doesn't just add more data; it adds a *second, independent* constraint that the sum-only objective was never forced to respect, closing off almost every alternative concept assignment that a shortcut could hide in.

This is the same logical move as the concept-supervision mitigation in Section 5, generalized: a reasoning shortcut survives by exploiting slack in what the training objective actually checks. Anything that removes that slack -- extra labels, an extra task, a more constraining rule -- is a candidate mitigation, and multi-task learning here is a particularly elegant one because it doesn't require any new kind of label, just a second thing to compute from labels you already have.


## 4. Applying the idea: a driving scenario (transfer exercise)

Sections 2-4 were controlled demonstrations, each isolating one specific cause with a definition-satisfying check (task accuracy at ceiling, concept accuracy measured directly against known ground truth). This section is different in kind: it's a **transfer exercise**, taking the same intuition into a more realistic, safety-relevant perception setting to see whether it still bites. Read it as "does this idea generalize," not as a fourth controlled proof on equal footing with Sections 2-4.

Two hidden concepts, `pedestrian_present` and `light_is_red`, are each observed only through a noisy sensor-like feature vector. The label follows a fixed rule: **stop if either concept is true.** In the training data, the two concepts are **confounded**: they always co-occur (`pedestrian_present == light_is_red` for every training example) -- a real-world analogue of a dataset collected somewhere a pedestrian is almost always accompanied by a red light, through correlation rather than causation. One sensor (the traffic-light one) is also deliberately noisier than the other, which reliably determines which of the two encoders below ends up doing all the work.

**Being honest about what this does and doesn't establish:** if a model simply uses one *raw input feature* instead of another because they're correlated, that's ordinary spurious-correlation shortcut learning -- a well-known failure mode with or without any symbolic layer involved, and a skeptical reader would be right to ask whether that's all this is. The mechanism actually at work here is a little more specific: because `OR(a, b)` barely changes with `b` once `a` is already confidently `1`, the `light_is_red` encoder's gradient signal vanishes almost entirely whenever the `pedestrian_present` encoder is already confident -- not because the raw light features are uninformative (a direct classifier trained on them alone would separate the classes just fine), but because the *symbolic combination rule* stops rewarding it for trying. That's a concept-level effect, specific to the symbolic combiner, layered on top of an ordinary data confound -- but we're presenting it as a plausible, illustrative mechanism for how a NeSy shortcut could arise from data structure in practice, not as a rigorously isolated proof of concept-level RS the way Section 3's entangled-vs-disentangled comparison was.


In [ ]:
class BinaryConceptEncoder(nn.Module):
    """Turns a noisy sensor reading into a belief distribution over a
    binary concept (absent=0, present=1).
    """

    def __init__(self, input_dim=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, features):
        return F.softmax(self.net(features), dim=-1)


def symbolic_or_distribution(concept1_probs, concept2_probs):
    """The fixed stop/go rule: stop if EITHER concept is present.
    P(stop) = 1 - P(both absent), assuming independence.
    """
    probability_both_absent = concept1_probs[:, 0] * concept2_probs[:, 0]
    probability_stop = 1 - probability_both_absent
    return torch.stack([1 - probability_stop, probability_stop], dim=1)


def make_confounded_driving_data(num_examples, seed=0, pedestrian_noise=0.3, light_noise=1.2):
    """Training data where pedestrian_present and light_is_red are the
    SAME underlying event -- perfectly confounded. The light sensor is
    also noisier, which reliably (not by luck of initialization)
    determines which encoder ends up starved of useful gradient: the
    cleaner pedestrian signal gets there first.
    """
    rng = np.random.default_rng(seed)
    shared_event = rng.integers(0, 2, size=num_examples)
    stop_label = shared_event.copy()

    def noisy_features(concept, noise):
        signal = np.stack([concept, 1 - concept], axis=1).astype(np.float32)
        return signal + rng.normal(0, noise, size=(num_examples, 2))

    return (
        torch.tensor(noisy_features(shared_event, pedestrian_noise), dtype=torch.float32),
        torch.tensor(noisy_features(shared_event, light_noise), dtype=torch.float32),
        torch.tensor(shared_event, dtype=torch.long),
        torch.tensor(shared_event, dtype=torch.long),
        torch.tensor(stop_label, dtype=torch.long),
    )


def make_decorrelated_driving_data(num_examples, seed=1):
    """The distribution-shift test set: pedestrian and light are forced
    to DISAGREE, a combination the confounded training data never
    contained. This is where a model that only ever tracked one of the
    two concepts gets exposed.
    """
    rng = np.random.default_rng(seed)
    pedestrian_present = rng.integers(0, 2, size=num_examples)
    light_is_red = 1 - pedestrian_present
    stop_label = (pedestrian_present | light_is_red).astype(np.int64)

    def noisy_features(concept, noise):
        signal = np.stack([concept, 1 - concept], axis=1).astype(np.float32)
        return signal + rng.normal(0, noise, size=(num_examples, 2))

    return (
        torch.tensor(noisy_features(pedestrian_present, 0.3), dtype=torch.float32),
        torch.tensor(noisy_features(light_is_red, 1.2), dtype=torch.float32),
        torch.tensor(pedestrian_present, dtype=torch.long),
        torch.tensor(light_is_red, dtype=torch.long),
        torch.tensor(stop_label, dtype=torch.long),
    )


In [ ]:
def train_driving_model(pedestrian_features, light_features, pedestrian_labels, light_labels, stop_labels,
                         epochs=800, concept_supervision_weight=0.0, lr=1e-2):
    """Trains both concept encoders using only the stop/go label,
    unless concept_supervision_weight > 0, which adds a direct loss on
    the true concept labels -- the mitigation strategy for this section.
    """
    pedestrian_encoder = BinaryConceptEncoder()
    light_encoder = BinaryConceptEncoder()
    optimizer = torch.optim.Adam(
        list(pedestrian_encoder.parameters()) + list(light_encoder.parameters()), lr=lr
    )

    for epoch in range(epochs):
        pedestrian_probs = pedestrian_encoder(pedestrian_features)
        light_probs = light_encoder(light_features)
        stop_probs = symbolic_or_distribution(pedestrian_probs, light_probs)

        loss = F.nll_loss(torch.log(stop_probs + 1e-8), stop_labels)
        if concept_supervision_weight > 0:
            concept_loss = (
                F.nll_loss(torch.log(pedestrian_probs + 1e-8), pedestrian_labels)
                + F.nll_loss(torch.log(light_probs + 1e-8), light_labels)
            )
            loss = loss + concept_supervision_weight * concept_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return pedestrian_encoder, light_encoder


def evaluate_driving_model(pedestrian_encoder, light_encoder, pedestrian_features, light_features,
                            pedestrian_labels, light_labels, stop_labels):
    """Reports task accuracy plus PER-CONCEPT F1 -- separately for
    pedestrian and light, not averaged together -- because averaging
    a healthy concept with a collapsed one hides the collapse.
    """
    with torch.no_grad():
        pedestrian_probs = pedestrian_encoder(pedestrian_features)
        light_probs = light_encoder(light_features)
        predicted_stop = symbolic_or_distribution(pedestrian_probs, light_probs).argmax(dim=1)

    return {
        "task_accuracy": float((predicted_stop == stop_labels).float().mean()),
        "pedestrian_f1": float(f1_score(pedestrian_labels.tolist(), pedestrian_probs.argmax(dim=1).tolist())),
        "light_f1": float(f1_score(light_labels.tolist(), light_probs.argmax(dim=1).tolist())),
    }


In [ ]:
confounded_train = make_confounded_driving_data(num_examples=500)
decorrelated_test = make_decorrelated_driving_data(num_examples=200)

pedestrian_encoder, light_encoder = train_driving_model(*confounded_train, concept_supervision_weight=0.0)

in_distribution_metrics = evaluate_driving_model(pedestrian_encoder, light_encoder, *confounded_train)
shift_metrics = evaluate_driving_model(pedestrian_encoder, light_encoder, *decorrelated_test)

print("no concept supervision")
print(f"  in-distribution: task_accuracy={in_distribution_metrics['task_accuracy']:.3f} "
      f"pedestrian_f1={in_distribution_metrics['pedestrian_f1']:.3f} light_f1={in_distribution_metrics['light_f1']:.3f}")
print(f"  decorrelated OOD: task_accuracy={shift_metrics['task_accuracy']:.3f} "
      f"pedestrian_f1={shift_metrics['pedestrian_f1']:.3f} light_f1={shift_metrics['light_f1']:.3f}")


In-distribution, task accuracy should sit at (or extremely close to) the ceiling this noisy problem allows -- and stay there even if you increase the training epochs further, which is what earns this the label "reasoning shortcut" rather than "still training." Alongside that ceiling-level task accuracy, `light_f1` should be far lower than `pedestrian_f1`, often near zero: the `light` encoder found a fully confident, fully deterministic, and fully *wrong* function (in this case, "the light is never red") that happens to satisfy the `OR` rule perfectly under the training confound. That's a misaligned semantics, not "meaningless" concepts -- the encoder isn't producing noise, it converged to a specific, wrong belief, confidently.

That misalignment is invisible if you only check in-distribution task accuracy. It becomes very visible on the decorrelated test set: task accuracy should drop sharply, because roughly half of those examples need the `light` concept to get the right answer, and that concept was never actually learned.


Now the mitigation. Direct concept supervision bypasses the `OR` gate entirely -- the concept loss doesn't route through it -- so it should recover the `light` concept regardless of the confound. But treat this as a genuine comparison, not a demonstration that "more supervision is strictly better": the paper is explicit that concept supervision can measurably *hurt* task-label performance even as it helps concepts, and mitigations in general involve real trade-offs rather than a free win. Use the slider to check both numbers, in both directions, rather than assuming supervision=1.0 is simply the "correct" setting.


In [ ]:
def run_driving_mitigation_experiment(concept_supervision_weight=0.0):
    pedestrian_encoder, light_encoder = train_driving_model(
        *confounded_train, concept_supervision_weight=concept_supervision_weight
    )
    in_distribution_metrics = evaluate_driving_model(pedestrian_encoder, light_encoder, *confounded_train)
    shift_metrics = evaluate_driving_model(pedestrian_encoder, light_encoder, *decorrelated_test)

    print(f"concept supervision weight = {concept_supervision_weight:.1f}")
    print(f"  in-distribution: task_accuracy={in_distribution_metrics['task_accuracy']:.3f} "
          f"pedestrian_f1={in_distribution_metrics['pedestrian_f1']:.3f} light_f1={in_distribution_metrics['light_f1']:.3f}")
    print(f"  decorrelated OOD: task_accuracy={shift_metrics['task_accuracy']:.3f} "
          f"pedestrian_f1={shift_metrics['pedestrian_f1']:.3f} light_f1={shift_metrics['light_f1']:.3f}")


widgets.interact_manual(
    run_driving_mitigation_experiment,
    concept_supervision_weight=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.0, description="supervision"),
)


Try 0.0 and 1.0 first, then something in between. `light_f1` and decorrelated-OOD task accuracy should both climb substantially as supervision increases -- but check whether in-distribution task accuracy stays exactly where it was, or dips slightly. If it dips, that's not a bug in this notebook; it's the trade-off the paper describes, made visible.


## 5. Evaluation checklist

The question "did the model work?" is really several separate questions. Before trusting a NeSy model's concepts, check:

- [ ] **Is task accuracy at (or extremely close to) its ceiling?** If not, stop -- you may be looking at an optimization problem, not a reasoning shortcut. (Section 3.)
- [ ] **Does the rule, or the architecture, or the data structurally permit an alternative solution?** Section 1 showed a rule that does (XOR); Section 2 showed that the *same* rule and data can be shortcut-prone or not, purely depending on architecture.
- [ ] **Is "constraint satisfaction" actually informative here, or is it guaranteed by construction?** If the symbolic rule is applied directly to the network's own beliefs, it's the latter -- a plumbing check, not a validation of the concepts flowing through it.
- [ ] **Is concept F1 checked per concept, not just averaged?** Section 4 showed that averaging a healthy concept together with a collapsed one hides the collapse.
- [ ] **Does task accuracy survive a distribution shift** that specifically breaks a confound present in training?
- [ ] **If a mitigation improved concept quality, did anything else get worse?** Treat "more supervision" as a trade-off to measure, not a free win to assume.

A model that only passes the first check has not demonstrated that it learned the concepts you intended -- only that it found *a* solution that fits what it has seen so far, and calling its concepts "meaningless" would overstate the case: they're better described as **misaligned with the semantics we intended**, often confidently and consistently so. This notebook demonstrated that gap taking two rigorously verified shapes -- a rule-level symmetry that no dataset size can break (Section 1), and an architecture that reintroduces ambiguity a different architecture had already ruled out on identical data (Section 2) -- plus one shape that only *looks* like a shortcut but is really an undertrained model (Section 3), and one transfer exercise suggesting the same intuition can show up in a more realistic, data-driven form (Section 4), with the caveat that Section 4 is illustrative rather than as tightly proven as Sections 1-2. Telling all of these apart, rather than assuming the first plausible story, is most of the work.


## 6. Optional appendix (future notebook)

This notebook implemented the symbolic rules directly in PyTorch so the mechanics stay transparent. A natural follow-on notebook would introduce **DeepProbLog**, one of the NeSy architectures studied directly in Marconato et al. (2023), and repeat these experiments using its probabilistic logic programming layer instead of the hand-written `symbolic_sum_distribution` / `symbolic_or_distribution` functions above -- showing that these phenomena aren't an artifact of this notebook's specific implementation, but a property of the neurosymbolic setup itself. A second natural extension would be to actually reproduce the paper's MNIST-AddMul multi-task result from Section 3 end to end (this notebook cites their numbers rather than re-deriving them, since our own attempt to reproduce it from scratch ran into real optimization difficulties that a short teaching notebook wasn't the right place to fully resolve).
